In [ ]:
from IPython import display
import ultralytics

In [ ]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
import cv2
import matplotlib.pyplot as plt

In [ ]:
display.clear_output()
ultralytics.checks()

In [ ]:
model = ultralytics.YOLO("yolo26n.yaml").load("yolo26n.pt")

In [ ]:
# Rec (bbox) - Stage 1: freeze backbone, train head and neck
model.train(data="custom2.yaml",
            epochs=50, batch=-1,
            imgsz=640,
            freeze=10,
            name="rec_stage1", exist_ok=True,
            workers=2,
            )

# Stage 2: unfreeze all, fine-tune with lower lr
# model = ultralytics.YOLO("runs/detect/rec_stage1/weights/best.pt")
# model.train(data="custom2.yaml", epochs=30, optimizer="AdamW", lr0=0.001, name="rec_stage2", exist_ok=True)

In [ ]:
model = ultralytics.YOLO("yolo26n-seg.yaml").load("yolo26n-seg.pt")

In [ ]:
# Poly (segment) - Stage 1: freeze backbone, train head and neck
model.train(data="custom3.yaml",
            epochs=50, batch=-1,
            imgsz=640,
            freeze=10,
            name="poly_stage1", exist_ok=True,
            workers=2,
            )

# Stage 2: unfreeze all, fine-tune with lower lr
# model = ultralytics.YOLO("runs/segment/poly_stage1/weights/best.pt")
# model.train(data="custom3.yaml", epochs=30, optimizer="AdamW", lr0=0.001, name="poly_stage2", exist_ok=True)

In [ ]:
# สรุปผลรอบสุดท้ายจาก results.csv
import csv

for task, name in (("detect", "rec_stage1"), ("segment", "poly_stage1")):
    with open(f"runs/{task}/{name}/results.csv", newline="", encoding="utf-8") as f:
        rows = [{k.strip(): v for k, v in r.items()} for r in csv.DictReader(f)]
    last = rows[-1]
    print(name, "epoch", last["epoch"])
    for k in last:
        if k.startswith("metrics/"):
            print(f"  {k:28s} {float(last[k]):.4f}")

In [ ]:
# เก็บผลส่งอาจารย์ไว้ที่ output/final (ไม่แตะ output เดิม)
import shutil
from pathlib import Path

OUT = Path("output/final")


def export(task, name, tag, weight_name):
    run = Path("runs") / task / name
    (OUT / "weights").mkdir(parents=True, exist_ok=True)
    shutil.copy2(run / "weights" / "best.pt", OUT / "weights" / weight_name)
    dst = OUT / "train_results" / tag
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(run, dst, ignore=shutil.ignore_patterns("weights"))
    print("exported", name)


export("detect", "rec_stage1", "rec", "best_rec_detect.pt")
export("segment", "poly_stage1", "poly", "best_poly_segment.pt")